# CardioIA — diagnóstico visual acadêmico com MLP

Notebook do **Ir Além 2**. O objetivo é classificar imagens de ECG em **normal** e **anormal** usando uma rede Perceptron Multicamadas implementada com Keras.

> O experimento é educacional, não possui validação clínica e não produz diagnóstico.

## 1. Governança antes do treinamento

A amostra da Fase 1 possui 120 imagens, com 30 exames por classe original. Os hashes garantem que não há cópias binárias na amostra.

A fonte não fornece identificador de paciente. Portanto, conseguimos impedir que o mesmo exame duplicado apareça em treino e teste, mas **não podemos garantir separação por indivíduo**. O balanceamento da amostra também não representa prevalência real.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "fase2").exists():
    RAIZ = Path.cwd().parents[1]
sys.path.insert(0, str(RAIZ / "fase2" / "src"))

from treinar_mlp_ecg import (
    carregar_pixels,
    criar_mlp,
    dividir_dados,
    inventariar_imagens,
    treinar_mlp,
)

## 2. Inventário e classificação binária

A classe `normal` permanece normal. Infarto do miocárdio, histórico de infarto e batimento anormal são agrupados como `anormal`, sem combinar estes exames com os registros tabulares ou textuais.

In [ ]:
inventario = inventariar_imagens()
display(pd.crosstab(inventario["classe_original"], inventario["classe_binaria"]))
display(pd.DataFrame({
    "total": [len(inventario)],
    "hashes_unicos": [inventario["hash_sha256"].nunique()],
    "duplicadas": [int(inventario["hash_sha256"].duplicated().sum())],
}))

## 3. Pré-processamento

Cada imagem é convertida para tons de cinza, redimensionada para 64 × 64 pixels, normalizada para o intervalo 0–1 e achatada em um vetor com 4.096 entradas, formato adequado para uma MLP.

In [ ]:
pixels = carregar_pixels(inventario)
display(pd.DataFrame({
    "amostras": [pixels.shape[0]],
    "atributos_por_imagem": [pixels.shape[1]],
    "valor_minimo": [pixels.min()],
    "valor_maximo": [pixels.max()],
}))

## 4. Divisão estratificada e verificação de vazamento

A divisão utiliza 80% dos exames para treino e 20% para teste. A estratificação preserva a proporção das classes binárias.

In [ ]:
x_treino, x_teste, y_treino, y_teste, treino_idx, teste_idx = dividir_dados(
    inventario,
    pixels,
)
assert not set(inventario.iloc[treino_idx]["hash_sha256"]).intersection(
    inventario.iloc[teste_idx]["hash_sha256"]
)
display(pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "imagens": [len(y_treino), len(y_teste)],
    "normais": [(y_treino == 0).sum(), (y_teste == 0).sum()],
    "anormais": [(y_treino == 1).sum(), (y_teste == 1).sum()],
}))

## 5. Arquitetura MLP com Keras

A rede recebe o vetor de pixels, utiliza duas camadas densas com ativação ReLU, dropout para regularização e uma saída sigmoide para classificação binária.

In [ ]:
modelo_demonstracao = criar_mlp(64 * 64)
modelo_demonstracao.summary()

## 6. Treinamento e avaliação

São usados pesos de classe para reduzir o efeito do desbalanceamento binário. O early stopping restaura os melhores pesos observados na validação.

In [ ]:
modelo, historico, inventario, metricas, previsoes, y_teste, y_predito = treinar_mlp()
resumo = {
    chave: valor
    for chave, valor in metricas.items()
    if chave != "relatorio"
}
display(pd.DataFrame([resumo]).style.format({
    "acuracia": "{:.3f}",
    "acuracia_balanceada": "{:.3f}",
    "precisao_anormal": "{:.3f}",
    "recall_anormal": "{:.3f}",
    "f1_anormal": "{:.3f}",
    "roc_auc": "{:.3f}",
}))
display(pd.DataFrame(metricas["relatorio"]).T)

In [ ]:
pd.DataFrame(historico.history)[["loss", "val_loss"]].plot(
    title="Perda durante o treinamento"
)
plt.xlabel("Época")
plt.ylabel("Binary cross-entropy")
plt.show()

display(previsoes.head(10))

## 7. Conclusão responsável

A atividade demonstra pré-processamento de imagens, implementação de MLP com Keras, treinamento e avaliação. Os resultados devem ser lidos apenas como desempenho nesta pequena amostra curada.

Limitações centrais:

- ausência de identificador de paciente;
- apenas 120 imagens;
- classes originais artificialmente balanceadas;
- origem populacional e tecnológica específica;
- MLP sobre pixels achatados, sem arquitetura especializada em visão;
- ausência de validação externa, prospectiva ou clínica.